In [4]:
import os
print(os.getcwd())

C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\notebook\Advanced_Analytics.ipynb


In [7]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nFolders in current location:")
print(os.listdir())

Current Working Directory:
C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\notebook\Advanced_Analytics.ipynb

Folders in current location:
['.ipynb_checkpoints', 'Untitled.ipynb']


In [8]:
import os

print(os.getcwd())

C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\notebook\Advanced_Analytics.ipynb


In [9]:
import os

print(os.listdir(".."))

['Advanced_Analytics.ipynb', 'EDA_Analysis.ipynb', 'Performance_Analytics.ipynb']


In [10]:
import pandas as pd

nav = pd.read_csv(
    r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\Data\processed\nav_history_clean.csv"
)

print(nav.head())

   amfi_code        date       nav
0     100016  2022-01-03  520.4608
1     100016  2022-01-04  515.0971
2     100016  2022-01-05  521.7239
3     100016  2022-01-06  515.7880
4     100016  2022-01-07  515.1639


In [11]:
import numpy as np

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
    .pct_change()
)

var_cvar = nav.groupby(
    "amfi_code"
)["daily_return"].apply(
    lambda x: pd.Series({
        "VaR_95": np.percentile(
            x.dropna(),
            5
        ),
        "CVaR_95": x[
            x <= np.percentile(
                x.dropna(),
                5
            )
        ].mean()
    })
).unstack()

var_cvar.reset_index(
    inplace=True
)

var_cvar.head()

,amfi_code,VaR_95,CVaR_95
0,100016,-0.014364,-0.018060
1,100025,-0.003793,-0.004994
2,100033,-0.019034,-0.023456
3,101206,-0.013282,-0.017439
4,101207,-0.026021,-0.032459


In [12]:
var_cvar.to_csv(
    r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\reports\var_cvar_report.csv",
    index=False
)

In [13]:
transactions = pd.read_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\Data\processed\investor_transactions_clean.csv"
)

performance = pd.read_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\Data\processed\scheme_performance_clean.csv"
)

holdings = pd.read_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\Data\Raw\09_portfolio_holdings.csv"
)

In [15]:
print(transactions.columns.tolist())
print(performance.columns.tolist())
print(holdings.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']
['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct', 'market_value_cr', 'current_price_inr', 'portfolio_date']


In [21]:
print(transactions.columns.tolist())

['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [24]:
import matplotlib.pyplot as plt
import numpy as np

risk_free = 0.065 / 252

funds = nav["amfi_code"].unique()[:5]

print(funds)

[100016 100025 100033 101206 101207]


In [25]:
fund = funds[0]

temp = nav[nav["amfi_code"] == fund].copy()

temp["daily_return"] = temp["nav"].pct_change()

temp["rolling_sharpe"] = (
    (temp["daily_return"].rolling(90).mean() - risk_free)
    /
    temp["daily_return"].rolling(90).std()
) * np.sqrt(252)

temp[["date","rolling_sharpe"]].head()

,date,rolling_sharpe
0,2022-01-03,NaN
1,2022-01-04,NaN
2,2022-01-05,NaN
3,2022-01-06,NaN
4,2022-01-07,NaN


In [26]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

transactions["cohort_year"] = (
    transactions.groupby("investor_id")
    ["transaction_date"]
    .transform("min")
    .dt.year
)

cohort = transactions.groupby(
    "cohort_year"
).agg(
    avg_sip=("amount_inr","mean"),
    total_invested=("amount_inr","sum")
)

cohort

,avg_sip,total_invested
cohort_year,,
2024,107422.541832,3491125187
2025,109158.577061,30455243


In [27]:
cohort.to_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\reports\cohort_analysis.csv"
)

In [28]:
transactions = transactions.sort_values(
    ["investor_id","transaction_date"]
)

transactions["gap_days"] = (
    transactions.groupby("investor_id")
    ["transaction_date"]
    .diff()
    .dt.days
)

continuity = transactions.groupby(
    "investor_id"
).agg(
    avg_gap=("gap_days","mean"),
    txn_count=("gap_days","count")
)

continuity["status"] = np.where(
    continuity["avg_gap"] > 35,
    "At Risk",
    "Healthy"
)

continuity.head()

,avg_gap,txn_count,status
investor_id,,,
INV000001,38.000,2,At Risk
INV000002,82.800,5,At Risk
INV000003,238.000,1,At Risk
INV000004,53.375,8,At Risk
INV000005,52.000,7,At Risk


In [29]:
top_funds = (
    performance
    .sort_values(
        "sharpe_ratio",
        ascending=False
    )
    [["scheme_name","risk_grade","sharpe_ratio"]]
    .head(10)
)

top_funds

,scheme_name,risk_grade,sharpe_ratio
14,ICICI Pru Liquid Fund - Regular - Growth,Low,7.68
23,Kotak Liquid Fund - Regular - Growth,Low,6.18
30,ABSL Liquid Fund - Regular - Growth,Low,5.14
9,HDFC Short Term Debt Fund - Regular - Growth,Low,1.84
4,SBI Magnum Gilt Fund - Regular Plan - Growth,Low,1.52
19,Nippon India Gilt Securities Fund - Regular - ...,Low,1.33
34,Mirae Asset Large Cap Fund - Regular - Growth,Moderate,1.06
5,HDFC Top 100 Fund - Regular Plan - Growth,Moderate,1.06
11,ICICI Pru Bluechip Fund - Direct - Growth,Moderate,1.03
15,Nippon India Large Cap Fund - Regular - Growth,Moderate,1.00


In [30]:
hhi = holdings.groupby(
    "amfi_code"
)["weight_pct"].apply(
    lambda x: np.sum((x/100)**2)
)

hhi = hhi.reset_index()

hhi.columns = [
    "amfi_code",
    "HHI"
]

hhi.head()

,amfi_code,HHI
0,100016,0.139534
1,100033,0.147592
2,101206,0.129332
3,101207,0.200700
4,102885,0.174709


# Advanced Analytics Insights

### 1. Downside Risk Analysis
Funds such as AMFI 101207 exhibited the highest downside risk with a VaR(95%) of -2.60% and CVaR(95%) of -3.25%, indicating larger losses during adverse market conditions.

### 2. Investor Cohort Behaviour
The 2025 investor cohort invested a slightly higher average SIP amount (₹109,159) compared to the 2024 cohort (₹107,423), indicating growing investor confidence.

### 3. Cohort Contribution
Although average SIP amounts were similar, the 2024 cohort contributed significantly higher total investments (₹349.11 Cr) due to a larger investor base and longer participation period.

### 4. SIP Continuity Risk
Several investors were classified as "At Risk" due to average transaction gaps exceeding 35 days, highlighting potential SIP discontinuation risk.

### 5. Portfolio Concentration
Funds with HHI values above 0.18 demonstrated higher sector concentration, while funds with HHI closer to 0.10 showed greater diversification.

In [31]:
hhi.to_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\reports\hhi_report.csv",
index=False
)

In [32]:
continuity.to_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\reports\sip_continuity.csv"
)

In [33]:
cohort.to_csv(
r"C:\Users\ritik\OneDrive\Pictures\Desktop\Bluestock_mf_capstone\reports\cohort_analysis.csv"
)